# Module 17: Validation and Resampling

**Case:** How accurately can listing features predict an Airbnb price in
Hackney?

We use an untouched holdout set for final evaluation and cross-validation
within the remaining data for model comparison. The logic follows Gábor's
used-car cross-validation case and prepares the Airbnb LASSO workflow.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")

In [ ]:
airbnb = pd.read_csv("data/raw/airbnb_hackney_course.csv")
target = "price_eur"
numeric = ["accommodates", "bedrooms", "bathrooms", "minimum_nights", "number_of_reviews", "review_score"]
categorical = ["room_type", "property_type", "superhost", "instant_bookable", "kitchen", "wifi", "elevator"]
features = numeric + categorical
X = airbnb[features]
y = airbnb[target]

In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_train.shape, X_holdout.shape

The holdout set is now closed. We do not use it to choose variables,
transformations, or tuning values. Repeatedly checking it would turn it
into another training set.

In [ ]:
airbnb[[target] + numeric].describe().T.round(1)

In [ ]:
airbnb[features].isna().sum().sort_values(ascending=False).head(8)

## 1. Build reproducible preprocessing

In [ ]:
numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median"))])
categorical_pipe = Pipeline(
    [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocess = ColumnTransformer(
    [("numeric", numeric_pipe, numeric), ("categorical", categorical_pipe, categorical)]
)

In [ ]:
rich_model = Pipeline(
    [("preprocess", preprocess), ("model", LinearRegression())]
)

## 2. Compare a small baseline with the richer model

In [ ]:
baseline_features = ["accommodates"]
baseline = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
        ("model", LinearRegression()),
    ]
)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
baseline_scores = cross_validate(
    baseline, X_train[baseline_features], y_train, cv=cv,
    scoring="neg_root_mean_squared_error", return_train_score=True,
)
rich_scores = cross_validate(
    rich_model, X_train, y_train, cv=cv,
    scoring="neg_root_mean_squared_error", return_train_score=True,
)

In [ ]:
folds = pd.DataFrame(
    {
        "fold": range(1, 6),
        "baseline_rmse": -baseline_scores["test_score"],
        "rich_rmse": -rich_scores["test_score"],
    }
)
folds.round(2)

In [ ]:
folds[["baseline_rmse", "rich_rmse"]].agg(["mean", "std"]).round(2)

In [ ]:
plot_folds = folds.melt("fold", var_name="model", value_name="rmse")
fig, ax = plt.subplots(figsize=(8, 4))
sns.pointplot(data=plot_folds, x="fold", y="rmse", hue="model", ax=ax)
ax.set(title="Performance varies across validation folds", ylabel="RMSE (EUR)")
plt.show()

## 3. Evaluate the chosen workflow once

In [ ]:
rich_model.fit(X_train, y_train)
holdout_prediction = rich_model.predict(X_holdout)
holdout_rmse = root_mean_squared_error(y_holdout, holdout_prediction)
print(f"Final holdout RMSE: EUR {holdout_rmse:.2f}")

In [ ]:
holdout_results = pd.DataFrame({"observed": y_holdout, "predicted": holdout_prediction})
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=holdout_results, x="predicted", y="observed", alpha=0.6, ax=ax)
ax.plot([20, 400], [20, 400], linestyle="--", color="black")
ax.set(title="Final holdout predictions")
plt.show()

**Checkpoint:** Cross-validation estimates variation in performance across
plausible training samples. The untouched holdout answers a different
question: how did the selected workflow perform on data it never influenced?

Continue to Module 18, where regularization is tuned within the same
training data and evaluated on the same untouched holdout definition.

### Optional extension

Repeat the comparison for `log(price_eur)`. Back-transform the predictions
and evaluate RMSE in euros, not on the log scale alone.